1. IMPORTS AND SETUP

In [3]:
# Imports and setup 

import os
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

2. LOAD CLEANED ARTIFACTS

In [4]:
# Load the cleaned artifacts

BASE_DIR = "/workspaces/movie-recommender-system"
PROCESSED = os.path.join(BASE_DIR, "data", "processed")

train = pd.read_csv(os.path.join(PROCESSED, "train.csv"))
val = pd.read_csv(os.path.join(PROCESSED, "val.csv"))
test = pd.read_csv(os.path.join(PROCESSED, "test.csv"))

ratings_clean = pd.read_csv(os.path.join(PROCESSED, "ratings_clean.csv"))
movies_clean = pd.read_csv(os.path.join(PROCESSED, "movies_clean.csv"))
movie_stats = pd.read_csv(os.path.join(PROCESSED, "movie_stats.csv"))
genre_matrix = pd.read_csv(os.path.join(PROCESSED, "genre_encoded.csv"))
user_stats = pd.read_csv(os.path.join(PROCESSED, "user_stats.csv"))

# Convert datetime columns back to datetime if needed
for df, col in [(train, "rated_at"), (val, "rated_at"), (test, "rated_at"), (ratings_clean, "rated_at")]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print("Artifacts loaded successfully.")
print(f"Train shape: {train.shape}")
print(f"Val shape  : {val.shape}")
print(f"Test shape : {test.shape}")

Artifacts loaded successfully.
Train shape: (80419, 5)
Val shape  : (10059, 5)
Test shape : (10358, 5)


3. BUILD TRAIN-ONLY USER-ITEM MATRIX

In [5]:
# Build the recommender from train-only, so validation and test remain unseen

user_item_train = train.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print("Train user-item matrix created.")
print(f"Shape: {user_item_train.shape}")
print(f"Non-null ratings: {user_item_train.count().sum():,}")

Train user-item matrix created.
Shape: (610, 8227)
Non-null ratings: 80,419


4. CONVERT THE MATRIX TO SPARSE FORMAT

In [6]:
# Convert the matrix to sparse format

user_item_train_filled = user_item_train.fillna(0)

user_item_sparse = csr_matrix(user_item_train_filled.to_numpy())

user_ids = user_item_train.index.to_list()
movie_ids = user_item_train.columns.to_list()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}
idx_to_movie = {idx: movie_id for movie_id, idx in movie_to_idx.items()}

print("Sparse matrix ready.")
print(f"CSR shape: {user_item_sparse.shape}")
print(f"Stored values: {user_item_sparse.nnz:,}")


Sparse matrix ready.
CSR shape: (610, 8227)
Stored values: 80,419


5. FIT AN ITEM-ITEM KNN MODEL

In [7]:
# Compare movies to other movies based on the users who rated them

item_user_matrix = user_item_train_filled.T
item_user_sparse = csr_matrix(item_user_matrix.to_numpy())

knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=20,
    n_jobs=-1
)

knn_model.fit(item_user_sparse)

print("Item-item KNN model trained.")
print(f"Item-user matrix shape: {item_user_matrix.shape}")

Item-item KNN model trained.
Item-user matrix shape: (8227, 610)


6. BUILD A POPULARITY FALLBACK RECOMMENDER

In [8]:
# Useful for new users or users with very little rating history

popularity_df = (
    train.groupby("movieId")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

global_mean = train["rating"].mean()
vote_weight = popularity_df["rating_count"].mean()

popularity_df["weighted_score"] = (
    (popularity_df["rating_count"] / (popularity_df["rating_count"] + vote_weight)) * popularity_df["avg_rating"]
    + (vote_weight / (popularity_df["rating_count"] + vote_weight)) * global_mean
)

popularity_df = popularity_df.merge(
    movies_clean[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

popularity_df = popularity_df.sort_values("weighted_score", ascending=False).reset_index(drop=True)

print("Popularity fallback ready.")
print(popularity_df.head(10))


Popularity fallback ready.
   movieId  avg_rating  rating_count  weighted_score  \
0      318    4.416961           283        4.386784   
1      858    4.302857           175        4.261077   
2      750    4.304598            87        4.224650   
3     2959    4.256410           195        4.220928   
4    48516    4.287356            87        4.209151   
5     1204    4.381579            38        4.203884   
6     1197    4.251938           129        4.199896   
7     1196    4.233161           193        4.198449   
8     1221    4.252212           113        4.193366   
9      260    4.218884           233        4.190467   

                                               title  \
0                   Shawshank Redemption, The (1994)   
1                              Godfather, The (1972)   
2  Dr. Strangelove or: How I Learned to Stop Worr...   
3                                  Fight Club (1999)   
4                               Departed, The (2006)   
5                   

7. FUNCTON TO GET SIMILAR MOVIES

In [9]:
# Function to get similar movies

def get_similar_movies(movie_id, n_neighbors=10):
    if movie_id not in movie_to_idx:
        return pd.DataFrame(columns=["movieId", "title", "genres", "similarity"])

    movie_idx = movie_to_idx[movie_id]

    distances, indices = knn_model.kneighbors(
        item_user_sparse[movie_idx],
        n_neighbors=n_neighbors + 1
    )

    neighbor_rows = []
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        neighbor_movie_id = idx_to_movie[idx]

        if neighbor_movie_id == movie_id:
            continue

        similarity = 1 - dist
        neighbor_rows.append((neighbor_movie_id, similarity))

    neighbors_df = pd.DataFrame(neighbor_rows, columns=["movieId", "similarity"])

    neighbors_df = neighbors_df.merge(
        movies_clean[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    neighbors_df = neighbors_df.sort_values("similarity", ascending=False).reset_index(drop=True)
    return neighbors_df

8. MAIN RECOMMENDATION FUNCTION FOR EXISTING USER

In [10]:
# Core recommender

def recommend_movies_for_user(user_id, top_n=10, neighbor_k=10, min_similarity=0.10):
    # Cold-start fallback
    if user_id not in user_to_idx:
        return popularity_df.head(top_n)[["movieId", "title", "genres", "weighted_score"]]

    # Get movies the user has already rated
    user_history = train[train["userId"] == user_id].copy()

    if user_history.empty:
        return popularity_df.head(top_n)[["movieId", "title", "genres", "weighted_score"]]

    rated_movie_ids = set(user_history["movieId"].tolist())

    # Candidate score accumulator
    candidate_scores = {}

    # Go through each movie the user rated
    for _, row in user_history.iterrows():
        seed_movie_id = row["movieId"]
        user_rating = row["rating"]

        similar_movies = get_similar_movies(seed_movie_id, n_neighbors=neighbor_k)

        for _, sim_row in similar_movies.iterrows():
            candidate_movie_id = sim_row["movieId"]
            similarity = sim_row["similarity"]

            if candidate_movie_id in rated_movie_ids:
                continue

            if similarity < min_similarity:
                continue

            weighted_contribution = similarity * user_rating

            if candidate_movie_id not in candidate_scores:
                candidate_scores[candidate_movie_id] = {
                    "weighted_sum": 0.0,
                    "similarity_sum": 0.0
                }

            candidate_scores[candidate_movie_id]["weighted_sum"] += weighted_contribution
            candidate_scores[candidate_movie_id]["similarity_sum"] += similarity

    # If no candidates found, use popularity fallback
    if not candidate_scores:
        fallback = popularity_df[~popularity_df["movieId"].isin(rated_movie_ids)]
        return fallback.head(top_n)[["movieId", "title", "genres", "weighted_score"]]

    # Convert candidate scores into predictions
    predictions = []
    for movie_id, values in candidate_scores.items():
        predicted_score = values["weighted_sum"] / values["similarity_sum"]
        predictions.append((movie_id, predicted_score))

    recs = pd.DataFrame(predictions, columns=["movieId", "predicted_score"])

    recs = recs.merge(
        movies_clean[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    recs = recs.merge(
        popularity_df[["movieId", "weighted_score", "rating_count"]],
        on="movieId",
        how="left"
    )

    recs = recs.sort_values(
        ["predicted_score", "weighted_score"],
        ascending=False
    ).head(top_n).reset_index(drop=True)

    return recs

9. TEST RECOMMENDER FOR ONE USER

In [11]:
# Select user from the training set to test

sample_user_id = train["userId"].iloc[0]

recommendations = recommend_movies_for_user(sample_user_id, top_n=10)

print(f"Recommendations for user {sample_user_id}:")
print(recommendations)

Recommendations for user 1:
   movieId  predicted_score                                    title  \
0     1252              5.0                         Chinatown (1974)   
1      904              5.0                       Rear Window (1954)   
2      778              5.0                     Trainspotting (1996)   
3      953              5.0             It's a Wonderful Life (1946)   
4      903              5.0                           Vertigo (1958)   
5     3334              5.0                         Key Largo (1948)   
6     1912              5.0                      Out of Sight (1998)   
7     2890              5.0                       Three Kings (1999)   
8     4571              5.0  Bill & Ted's Excellent Adventure (1989)   
9     3849              5.0              The Spiral Staircase (1945)   

                                genres  weighted_score  rating_count  
0     Crime|Film-Noir|Mystery|Thriller        4.135591            51  
1                     Mystery|Thrille

10. RECOMMEND SIMILAR MOVIES TO A SINGLE MOVIE

In [12]:
# Useful for the 'because you watched this' feature

movie_lookup = movies_clean[["movieId", "title"]].copy()

def find_movie_id_by_title(search_text):
    matches = movie_lookup[movie_lookup["title"].str.contains(search_text, case=False, na=False)]
    return matches.head(10)

# Example search
matches = find_movie_id_by_title("Toy Story")
print(matches)

# Example: if Toy Story movieId is found, use it here
example_movie_id = matches.iloc[0]["movieId"]
similar_movies = get_similar_movies(example_movie_id, n_neighbors=10)

print(f"\nMovies similar to movieId {example_movie_id}:")
print(similar_movies)

      movieId               title
0           1    Toy Story (1995)
2355     3114  Toy Story 2 (1999)
7355    78499  Toy Story 3 (2010)

Movies similar to movieId 1:
   movieId  similarity                                              title  \
0      780    0.570410               Independence Day (a.k.a. ID4) (1996)   
1      480    0.551900                               Jurassic Park (1993)   
2     3114    0.548202                                 Toy Story 2 (1999)   
3      260    0.547197          Star Wars: Episode IV - A New Hope (1977)   
4      356    0.527275                                Forrest Gump (1994)   
5     1210    0.527186  Star Wars: Episode VI - Return of the Jedi (1983)   
6      648    0.520158                         Mission: Impossible (1996)   
7     1580    0.517355                   Men in Black (a.k.a. MIB) (1997)   
8     4306    0.515959                                       Shrek (2001)   
9     1073    0.509495         Willy Wonka & the Chocolate Facto

11. SIMPLE EVALUATION ON VALIDATION DATA

In [13]:
# This checks if the recommender can recover movies rated in validation

def hit_rate_at_k(train_df, eval_df, top_n=10, min_val_rating=4.0, sample_users=100):
    eligible_users = []

    for user_id, user_eval in eval_df.groupby("userId"):
        if len(user_eval) == 0:
            continue

        if (user_eval["rating"] >= min_val_rating).sum() == 0:
            continue

        if user_id not in set(train_df["userId"].unique()):
            continue

        eligible_users.append(user_id)

    if len(eligible_users) == 0:
        return 0.0

    eligible_users = eligible_users[:sample_users]

    hits = 0
    total = 0

    for user_id in eligible_users:
        actual_positive_movies = set(
            eval_df[
                (eval_df["userId"] == user_id) &
                (eval_df["rating"] >= min_val_rating)
            ]["movieId"].tolist()
        )

        if not actual_positive_movies:
            continue

        recs = recommend_movies_for_user(user_id, top_n=top_n)
        recommended_movies = set(recs["movieId"].tolist())

        hit = len(actual_positive_movies.intersection(recommended_movies)) > 0
        hits += int(hit)
        total += 1

    return hits / total if total > 0 else 0.0

12. RUN VALIDATION AND TEST EVALUATION

In [14]:
# Run validation and test evaluation for quick performance measure

val_hit_rate_10 = hit_rate_at_k(train, val, top_n=10, min_val_rating=4.0, sample_users=100)
test_hit_rate_10 = hit_rate_at_k(train, test, top_n=10, min_val_rating=4.0, sample_users=100)

print(f"Validation Hit Rate@10: {val_hit_rate_10:.4f}")
print(f"Test Hit Rate@10      : {test_hit_rate_10:.4f}")


Validation Hit Rate@10: 0.2100
Test Hit Rate@10      : 0.1400


Explanation of results: 
The validation hit rate means: 21% of users had at least one correct recommendation in their top 10.
The test hit rate means: 14% of users had at least one correct recommendation in their top 10.

13. HELPER TO EXPLAIN RECOMMENDATIONS

In [15]:
# This helps see why a movie was recommended

def explain_recommendation(user_id, recommended_movie_id, neighbor_k=10):
    if user_id not in user_to_idx:
        print("User not found in training data.")
        return

    user_history = train[train["userId"] == user_id].copy()
    rated_movie_ids = set(user_history["movieId"].tolist())

    explanations = []

    for _, row in user_history.iterrows():
        seed_movie_id = row["movieId"]
        user_rating = row["rating"]

        similar_movies = get_similar_movies(seed_movie_id, n_neighbors=neighbor_k)

        match = similar_movies[similar_movies["movieId"] == recommended_movie_id]
        if not match.empty:
            sim = match.iloc[0]["similarity"]

            seed_title = movies_clean.loc[movies_clean["movieId"] == seed_movie_id, "title"].values
            seed_title = seed_title[0] if len(seed_title) > 0 else str(seed_movie_id)

            explanations.append({
                "because_you_rated": seed_title,
                "your_rating": user_rating,
                "similarity": sim
            })

    if len(explanations) == 0:
        print("No direct explanation found from nearest neighbors.")
        return

    explanation_df = pd.DataFrame(explanations).sort_values("similarity", ascending=False)
    print(explanation_df)

14. EXAMPLE FULL RUN

In [16]:
# Example full run using item-based collaborative filtering

# Pick a user
user_id_example = train["userId"].iloc[5]

# Get recommendations
recs = recommend_movies_for_user(user_id_example, top_n=5)
print(f"Top recommendations for user {user_id_example}:")
print(recs)

# Explain the first recommendation
if not recs.empty:
    rec_movie_id = recs.iloc[0]["movieId"]
    print(f"\nWhy movieId {rec_movie_id} was recommended:")
    explain_recommendation(user_id_example, rec_movie_id)


Top recommendations for user 1:
   movieId  predicted_score                         title  \
0     1252              5.0              Chinatown (1974)   
1      904              5.0            Rear Window (1954)   
2      778              5.0          Trainspotting (1996)   
3      953              5.0  It's a Wonderful Life (1946)   
4      903              5.0                Vertigo (1958)   

                             genres  weighted_score  rating_count  
0  Crime|Film-Noir|Mystery|Thriller        4.135591            51  
1                  Mystery|Thriller        4.090144            65  
2                Comedy|Crime|Drama        4.002538            85  
3    Children|Drama|Fantasy|Romance        3.979937            53  
4    Drama|Mystery|Romance|Thriller        3.894853            45  

Why movieId 1252 was recommended:
     because_you_rated  your_rating  similarity
0  Citizen Kane (1941)          5.0    0.451264


15. UPGRADED RECOMMENDER - PREPARE HYBRID LOOKUP TABLES

In [17]:
# Prepare hybrid lookup tables

# Section 15: Hybrid prep tables

import numpy as np
import pandas as pd

# 1) Keep only the genre columns
genre_cols = [c for c in genre_matrix.columns if c != "movieId"]

# 2) Fast lookup: movieId -> one-hot genre row
genre_lookup = genre_matrix.set_index("movieId")[genre_cols].copy()

# 3) Movie metadata lookup for readable outputs
movie_meta = (
    movies_clean[["movieId", "title", "genres"]]
    .drop_duplicates()
    .set_index("movieId")
)

# 4) Rebuild popularity from TRAIN only
popularity_df = (
    train.groupby("movieId")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

global_mean = train["rating"].mean()
vote_weight = popularity_df["rating_count"].mean()

popularity_df["weighted_score"] = (
    (popularity_df["rating_count"] / (popularity_df["rating_count"] + vote_weight)) * popularity_df["avg_rating"]
    + (vote_weight / (popularity_df["rating_count"] + vote_weight)) * global_mean
)

popularity_lookup = popularity_df.set_index("movieId")

print("Hybrid lookup tables ready.")
print("Genre columns:", len(genre_cols))
print("Popularity rows:", len(popularity_df))


Hybrid lookup tables ready.
Genre columns: 19
Popularity rows: 8227


16. BUILD USER GENRE PREFERENCE PROFILE

In [18]:
# User taste profile from liked movies

def build_user_genre_profile(user_id, train_df, min_rating=4.0):
    """
    Returns a normalized genre preference vector for one user.
    Uses only movies the user rated >= min_rating.
    """
    user_likes = train_df[
        (train_df["userId"] == user_id) &
        (train_df["rating"] >= min_rating)
    ][["movieId", "rating"]].copy()

    # Cold-start or no strong likes yet
    if user_likes.empty:
        return pd.Series(0.0, index=genre_cols)

    # Attach genre columns
    liked_with_genres = user_likes.merge(
        genre_matrix[["movieId"] + genre_cols],
        on="movieId",
        how="left"
    ).fillna(0)

    # Weight each liked movie’s genres by the user’s rating
    weighted_genres = liked_with_genres[genre_cols].multiply(
        liked_with_genres["rating"], axis=0
    ).sum()

    # Normalize so the largest genre preference becomes 1.0
    if weighted_genres.max() > 0:
        weighted_genres = weighted_genres / weighted_genres.max()

    return weighted_genres


def get_genre_overlap_score(movie_id, user_profile):
    """
    Measures how well a candidate movie's genres match the user's genre profile.
    Returns a value roughly between 0 and 1.
    """
    if movie_id not in genre_lookup.index:
        return 0.0

    movie_vec = genre_lookup.loc[movie_id].astype(float)

    # If the movie has no genre tags, no boost
    if movie_vec.sum() == 0:
        return 0.0

    overlap = np.dot(movie_vec.values, user_profile.values) / movie_vec.sum()
    return float(overlap)

17. HYBRID RECOMMENDER

In [19]:
# Hybrid recommender

existing_user_ids = set(train["userId"].unique())

def recommend_movies_hybrid(
    user_id,
    top_n=10,
    neighbor_k=15,
    min_similarity=0.10,
    min_rating_for_profile=4.0,
    genre_weight=0.35,
    popularity_weight=0.15
):
    """
    Hybrid recommender:
    1) Collaborative filtering candidate generation
    2) Genre-aware re-ranking
    3) Popularity tie-break / fallback
    """

    # 1. Cold-start user -> popularity fallback
    if user_id not in existing_user_ids:
        fallback = popularity_df.merge(
            movies_clean[["movieId", "title", "genres"]],
            on="movieId",
            how="left"
        ).sort_values("weighted_score", ascending=False)

        return fallback.head(top_n)[
            ["movieId", "title", "genres", "weighted_score"]
        ].reset_index(drop=True)

    # 2. Get this user's train history
    user_history = train[train["userId"] == user_id].copy()

    if user_history.empty:
        fallback = popularity_df.merge(
            movies_clean[["movieId", "title", "genres"]],
            on="movieId",
            how="left"
        ).sort_values("weighted_score", ascending=False)

        return fallback.head(top_n)[
            ["movieId", "title", "genres", "weighted_score"]
        ].reset_index(drop=True)

    rated_movie_ids = set(user_history["movieId"].tolist())

    # 3. Build user's genre taste profile
    user_profile = build_user_genre_profile(
        user_id=user_id,
        train_df=train,
        min_rating=min_rating_for_profile
    )

    # 4. Candidate accumulator
    candidate_scores = {}

    # 5. Generate candidate movies from baseline CF neighbors
    for _, row in user_history.iterrows():
        seed_movie_id = row["movieId"]
        user_rating = row["rating"]

        similar_movies = get_similar_movies(
            movie_id=seed_movie_id,
            n_neighbors=neighbor_k
        )

        for _, sim_row in similar_movies.iterrows():
            candidate_movie_id = sim_row["movieId"]
            similarity = sim_row["similarity"]

            if candidate_movie_id in rated_movie_ids:
                continue

            if similarity < min_similarity:
                continue

            weighted_contribution = similarity * user_rating

            if candidate_movie_id not in candidate_scores:
                candidate_scores[candidate_movie_id] = {
                    "weighted_sum": 0.0,
                    "similarity_sum": 0.0,
                    "reasons": []
                }

            candidate_scores[candidate_movie_id]["weighted_sum"] += weighted_contribution
            candidate_scores[candidate_movie_id]["similarity_sum"] += similarity
            candidate_scores[candidate_movie_id]["reasons"].append(
                (seed_movie_id, similarity, user_rating)
            )

    # 6. If no candidates found, fall back to popularity
    if not candidate_scores:
        fallback = popularity_df[
            ~popularity_df["movieId"].isin(rated_movie_ids)
        ].merge(
            movies_clean[["movieId", "title", "genres"]],
            on="movieId",
            how="left"
        ).sort_values("weighted_score", ascending=False)

        return fallback.head(top_n)[
            ["movieId", "title", "genres", "weighted_score"]
        ].reset_index(drop=True)

    # 7. Score candidates
    rows = []

    for candidate_movie_id, values in candidate_scores.items():
        cf_score = values["weighted_sum"] / values["similarity_sum"]

        genre_score = get_genre_overlap_score(
            movie_id=candidate_movie_id,
            user_profile=user_profile
        )

        pop_score = (
            popularity_lookup.loc[candidate_movie_id, "weighted_score"]
            if candidate_movie_id in popularity_lookup.index
            else global_mean
        )

        rating_count = (
            popularity_lookup.loc[candidate_movie_id, "rating_count"]
            if candidate_movie_id in popularity_lookup.index
            else 0
        )

        # Put popularity on roughly a 0-1 scale
        pop_score_norm = pop_score / 5.0

        hybrid_score = (
            cf_score
            + genre_weight * genre_score
            + popularity_weight * pop_score_norm
        )

        # Best explanation seed movie
        best_reason = sorted(
            values["reasons"],
            key=lambda x: x[1] * x[2],
            reverse=True
        )[0]

        reason_movie_id = best_reason[0]
        reason_title = (
            movie_meta.loc[reason_movie_id, "title"]
            if reason_movie_id in movie_meta.index
            else str(reason_movie_id)
        )

        rows.append({
            "movieId": candidate_movie_id,
            "cf_score": round(cf_score, 4),
            "genre_score": round(genre_score, 4),
            "popularity_score": round(pop_score, 4),
            "hybrid_score": round(hybrid_score, 4),
            "rating_count": int(rating_count),
            "reason": f"Because you liked: {reason_title}"
        })

    recs = pd.DataFrame(rows)

    recs = recs.merge(
        movies_clean[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    recs = recs.sort_values(
        ["hybrid_score", "cf_score", "popularity_score"],
        ascending=False
    ).head(top_n).reset_index(drop=True)

    return recs[
        [
            "movieId",
            "title",
            "genres",
            "cf_score",
            "genre_score",
            "popularity_score",
            "hybrid_score",
            "rating_count",
            "reason"
        ]
    ]

18. RUN THE HYBRID RECOMMENDER AND EVALUATE

In [20]:
# Quick test and evaluation

sample_user_id = train["userId"].iloc[0]

hybrid_recs = recommend_movies_hybrid(
    user_id=sample_user_id,
    top_n=10
)

print(f"Hybrid recommendations for user {sample_user_id}:")
display(hybrid_recs.head(10))

Hybrid recommendations for user 1:


,movieId,title,genres,cf_score,genre_score,popularity_score,hybrid_score,rating_count,reason
0,3030,Yojimbo (1961),Action|Adventure,5.0,0.9633,4.0356,5.4582,11,Because you liked: Duck Soup (1933)
1,3389,Let's Get Harry (1986),Action|Adventure,5.0,0.9633,3.3727,5.4383,1,Because you liked: Gulliver's Travels (1939)
2,1867,Tarzan and the Lost City (1998),Action|Adventure,5.0,0.9633,3.3727,5.4383,1,Because you liked: Gulliver's Travels (1939)
3,4636,"Punisher, The (1989)",Action,5.0,0.9266,3.2361,5.4214,3,"Because you liked: Ghost and the Darkness, The..."
4,3417,"Crimson Pirate, The (1952)",Adventure|Comedy,5.0,0.9006,3.4655,5.4192,1,Because you liked: Gulliver's Travels (1939)
5,1287,Ben-Hur (1959),Action|Adventure|Drama,5.0,0.8522,3.8097,5.4126,31,Because you liked: Planet of the Apes (1968)
6,4734,Jay and Silent Bob Strike Back (2001),Adventure|Comedy,5.0,0.9006,3.2267,5.4120,30,Because you liked: Dogma (1999)
7,1884,Fear and Loathing in Las Vegas (1998),Adventure|Comedy|Drama,5.0,0.8104,3.9108,5.4010,38,"Because you liked: Big Lebowski, The (1998)"
8,2471,Crocodile Dundee II (1988),Action|Adventure|Comedy,5.0,0.9093,2.7015,5.3993,22,Because you liked: Crocodile Dundee (1986)
9,2788,Monty Python's And Now for Something Completel...,Comedy,5.0,0.8012,3.9331,5.3984,24,Because you liked: Duck Soup (1933)


18b. EVALUATION FUNCTION

In [21]:
# Evaluation function

def hit_rate_at_k_hybrid(eval_df, top_n=10, min_eval_rating=4.0, sample_users=100):
    eligible_users = []

    for user_id, user_eval in eval_df.groupby("userId"):
        if (user_eval["rating"] >= min_eval_rating).sum() == 0:
            continue
        eligible_users.append(user_id)

    eligible_users = eligible_users[:sample_users]

    hits = 0
    total = 0

    for user_id in eligible_users:
        actual_positive_movies = set(
            eval_df[
                (eval_df["userId"] == user_id) &
                (eval_df["rating"] >= min_eval_rating)
            ]["movieId"].tolist()
        )

        if not actual_positive_movies:
            continue

        recs = recommend_movies_hybrid(user_id=user_id, top_n=top_n)
        recommended_movies = set(recs["movieId"].tolist())

        hit = len(actual_positive_movies.intersection(recommended_movies)) > 0
        hits += int(hit)
        total += 1

    return hits / total if total > 0 else 0.0


hybrid_val_hit_rate_10 = hit_rate_at_k_hybrid(val, top_n=10, min_eval_rating=4.0, sample_users=100)
hybrid_test_hit_rate_10 = hit_rate_at_k_hybrid(test, top_n=10, min_eval_rating=4.0, sample_users=100)

print(f"Hybrid Validation Hit Rate@10: {hybrid_val_hit_rate_10:.4f}")
print(f"Hybrid Test Hit Rate@10      : {hybrid_test_hit_rate_10:.4f}")

Hybrid Validation Hit Rate@10: 0.1200
Hybrid Test Hit Rate@10      : 0.0600


19. SAVE THE HYBRID_READY ARTIFACTS

In [22]:
# Saving extra hybrid artifacts

# Section 19: Save extra hybrid artifacts

hybrid_artifacts_dir = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(hybrid_artifacts_dir, exist_ok=True)

popularity_df.to_csv(os.path.join(hybrid_artifacts_dir, "popularity_df.csv"), index=False)

print("Saved hybrid artifacts:")
print(os.path.join(hybrid_artifacts_dir, "popularity_df.csv"))


Saved hybrid artifacts:
/workspaces/movie-recommender-system/data/processed/popularity_df.csv


In [23]:
train = pd.read_csv(os.path.join(PROCESSED, "train.csv"))
val = pd.read_csv(os.path.join(PROCESSED, "val.csv"))
test = pd.read_csv(os.path.join(PROCESSED, "test.csv"))
movies_clean = pd.read_csv(os.path.join(PROCESSED, "movies_clean.csv"))

print(train.shape, val.shape, test.shape, movies_clean.shape)

(80419, 5) (10059, 5) (10358, 5) (9742, 6)


PROJECT HANDOFF FOR RECOMMENDER SYSTEM

I completed the data preparation and baseline recommender preparation work. The processed files are saved in `data/processed/`.

Main files for the next phase:

* `train.csv` = training interactions
* `val.csv` = validation interactions
* `test.csv` = final test interactions
* `movies_clean.csv` = movie metadata for title/genre lookup

Additional supporting files:

* `ratings_clean.csv`
* `movie_stats.csv`
* `genre_encoded.csv`
* `user_stats.csv`
* `user_item_matrix.csv`
* `popularity_df.csv` (if needed for fallback/hybrid comparisons)

Recommended next step:
Use `train.csv` to build the matrix factorization model, tune on `val.csv`, and report final performance on `test.csv`.

Key columns:

* `userId`
* `movieId`
* `rating`
* `rated_at` (if present)
* `title` and `genres` are in `movies_clean.csv`

All preprocessing and cleaning were done before the split, and the outputs are ready to load directly into the next notebook/script.
